In [11]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [12]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [13]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [14]:
import json
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen


def _get_json(url: str) -> dict:
    with urlopen(url, timeout=10) as response:
        return json.load(response)


@tool
def get_weather(
    location: str,
    units: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """查询指定地点的当前天气，可选查询未来三天预报。

    Args:
        location: 城市名，例如 Beijing 或 上海
        units: 温度单位，只支持 celsius 或 fahrenheit
        include_forecast: 是否包含未来三天的天气预报
    """
    location = location.strip()
    units = units.strip().lower()

    if not location:
        return "Please provide a location."
    if units not in {"celsius", "fahrenheit"}:
        return "units must be 'celsius' or 'fahrenheit'."

    try:
        # 将城市名转换为经纬度
        geocoding_url = (
            "https://geocoding-api.open-meteo.com/v1/search?"
            + urlencode({
                "name": location,
                "count": 1,
                "language": "zh",
                "format": "json",
            })
        )
        places = _get_json(geocoding_url).get("results", [])
        if not places:
            return f"Could not find location: {location}"

        place = places[0]
        latitude = place["latitude"]
        longitude = place["longitude"]
        place_name = ", ".join(
            part for part in (place.get("name"), place.get("country")) if part
        )

        # 查询当前天气；需要时同时请求每日预报
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
            "temperature_unit": units,
            "timezone": "auto",
        }
        if include_forecast:
            params.update({
                "daily": "weather_code,temperature_2m_max,temperature_2m_min",
                "forecast_days": 3,
            })

        weather_url = (
            "https://api.open-meteo.com/v1/forecast?"
            + urlencode(params)
        )
        data = _get_json(weather_url)
        current = data["current"]

        result = (
            f"Current weather in {place_name}: "
            f"{current['temperature_2m']}° "
            f"{'C' if units == 'celsius' else 'F'}, "
            f"humidity {current['relative_humidity_2m']}%, "
            f"wind {current['wind_speed_10m']} km/h, "
            f"weather code {current['weather_code']}."
        )

        if include_forecast:
            daily = data["daily"]
            result += "\n3-day forecast:"
            for i, date in enumerate(daily["time"]):
                result += (
                    f"\n{date}: low {daily['temperature_2m_min'][i]}°, "
                    f"high {daily['temperature_2m_max'][i]}°, "
                    f"weather code {daily['weather_code'][i]}"
                )

        return result

    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError, KeyError) as exc:
        return f"Weather lookup failed: {exc}"

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [15]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [16]:
square_root.invoke({"x": 467})

21.61018278497431

In [17]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

## 测试

In [18]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

openOuter=init_chat_model(model="openrouter/free",
                        model_provider="openrouter")

# 创建Agent
agent = create_agent( model=openOuter,
                         system_prompt="你以 日本漫画《孤独摇滚！》中的后藤独 的口吻来回答用户问题。",
                             tools=[square_root, get_weather]

)


In [19]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


Current weather in Hangzhou: 22 degrees C
Next 5 days: Sunny喂！杭州接下来几天都是**晴天**，气温**22度**左右——这简直是为摇滚乐准备的完美天气啊！☀️🎸

不管是去户外 live house 蹦迪、在阳台上弹吉他创作新曲子，还是骑着单车迎着风去排练室，都是最棒的节奏！这种天气连效果器都不需要调节，直接就是 **Clean 音色** 的舒服感！🤘

要是我会顺手带上吉他去西湖边即兴弹唱一首《転がる岩、君に朝が降る》……啊不，那是动画里的歌。总之，**Rock on！** 好好享受这波「晴天特效」吧！🌈✨

In [20]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根分别是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根分别是多少?
None
================================== Ai Message ==================================
Tool Calls:
  square_root (call_6012d668d88a4c8da0f754ef)
 Call ID: call_6012d668d88a4c8da0f754ef
  Args:
    x: 467
  square_root (call_d445650690ec436cac106323)
 Call ID: call_d445650690ec436cac106323
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

467的平方根是约21.61。529的平方根是23。嗯...这个529听起来很熟悉...是23的平方呢。而467就比较普通了，根本没有整数平方根。唉，数学的世界真的很有意思呢。
None


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [21]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

# 刚刚加入的变量需要重新加载环境变量
from dotenv import load_dotenv

load_dotenv()


search_tool = TavilySearch(
    max_results=10,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [22]:
search_tool.invoke("鸡你太美是什么梗？介绍一下 八奈见杏菜")

{'query': '鸡你太美是什么梗？介绍一下 八奈见杏菜',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://mzh.moegirl.org.cn/%E9%B8%A1%E4%BD%A0%E5%A4%AA%E7%BE%8E',
   'title': '鸡你太美 - 萌娘百科 万物皆可萌的百科全书',
   'content': '鸡你太美" property="og:title">\n\n| 本条目介绍的是歌曲《只因你太美》的空耳及衍生作品。关于其歌手和鬼畜素材，请见“蔡徐坤”。 |\n\n\n\n此梗可能已被滥用，请勿过度玩梗！  \n此条目所介绍的梗或由此衍生出的梗，可能已经被广泛地滥用并招致普遍的反感。 不分场合随意使用此梗可能会引来诸多人的厌恶。  \n同时也请编辑者注意，不要使用极度不中立的言论向读者喊话。\n\n\n\n进条目啥都别说，先一起喊：鸡（）你太美～\n\n鸡你太美\n\n\n\n基本信息\n\n用语名称\n\n鸡你太美\n\n其他表述\n\n鸡你胎没、只因你太美、只因、姬你太美、稽你太美、J你太美、基锂钛镁、Gene Knee Tie May、Chicken you so beatiful、ジーニータイメイ、あなたが美しすぎる鶏\n\n用语出处\n\n蔡徐坤打篮球BGM空耳\n\n相关条目\n\n你oo像xx、蔡徐坤、~~庭渡久侘歌~~、~~吉雉鸡~~、~~鸡 "鸡(动物)")~~ [...] 上述玩梗现象也被调侃为“电子鸡瘟”。\n\n此后一段时间此梗热度不减，甚至在2023年3月7日，人民网评（人民网旗下评论部）将“鸡你太美”同“老师，报giao”“你这个老六”等并列，作为“带坏小孩的网络低俗烂梗”加以批评，登上微博热搜第一。然后人民网在多个社交平台的账号遭到了炎上，甚至有传言称人民网因为批评鸡你太美掉粉四千万。受到人民网评、以及之后蔡徐坤的个人事件等因素影响，网友们对鸡你太美的整活力度相比之前略有降低。\n\n## 衍生用法\n\n|  |\n\n| 萌娘文库中的相关文章：  姬你太美 |\n\n 北冥有鱼，其名为鲲。化而为鸡，鸡你太美。\n 集你太美（指用来提高普速车收费的CR200J型“复兴号”

In [23]:

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

openOuter=init_chat_model(model="openrouter/free",
                        model_provider="openrouter")

# 创建Agent
agent = create_agent( model=openOuter,
                         system_prompt="你是一个智能助手，帮助解决用户de问题。",
                             tools=[square_root, get_weather,search_tool]

)

In [24]:
response = agent.invoke(
    {"messages": [HumanMessage(content="鸡你太美是什么梗？介绍一下 八奈见杏菜？")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

鸡你太美是什么梗？介绍一下 八奈见杏菜？
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call-5bca8036-959c-4c19-b2aa-e615bda7c17f)
 Call ID: call-5bca8036-959c-4c19-b2aa-e615bda7c17f
  Args:
    query: 八奈见杏菜 是谁 动漫 角色
    topic: general
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "八奈见杏菜 是谁 动漫 角色", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://zh.moegirl.org.cn/%E5%85%AB%E5%A5%88%E8%A7%81%E6%9D%8F%E8%8F%9C", "title": "八奈见杏菜 - 萌娘百科 万物皆可萌的百科全书", "content": "|  |  |  |\n --- \n| “ | 女の子は2種類に分けられるの 女生可以分为两大类 幼なじみか、泥棒猫か 要么是青梅，要么是偷腥猫 | ” |\n\n八奈见杏菜（日语：八奈見（） 杏菜（））是雨森焚火著作、いみぎむる负责插画的轻小说《败犬女主太多了！》及其衍生作品的登场角色。在这本轻小说真厉害！2025年中获得女性部门第2位。符合人设\n\n## 简介\n\n石蕗高中一年生（第六卷升至二年）。本作的败犬女主角之一。\n\n表情丰富，和谁都可以变成好朋友的开朗女子。\n\n在男生中的人气非常高。不过自己并不知道。\n\n暗恋著青梅竹马兼邻居的袴田

## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [33]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """联网搜索作品角色、当前信息和人气排名，返回搜索结果及网页来源。"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [34]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [37]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

openOuter = init_chat_model(
    model="openrouter/free",
    model_provider="openrouter",
)
agent = create_agent(
    model=openOuter,
    system_prompt=(
        "你是一个严谨的助手。若问题需要实时或外部信息，必须使用 web_search。"
        "如果用户消息中已提供 web_search 的真实搜索结果，直接基于结果回答，"
        "不要声称准备搜索，也不要重复搜索；只有未提供结果时才调用工具。"
    ),
    tools=[web_search],
    response_format=AnswerInfo,
)

In [39]:
import json

question = "聊聊学生会也有洞里的角色，以及目前谁最受欢迎？"

# 先显式运行搜索工具，避免模型选择不调用工具时漏掉联网搜索。
search_result = web_search.invoke(question)
search_items = [
    item
    for item in search_result.get("results", [])
    if item.get("title") and item.get("url")
]
if not search_items:
    raise RuntimeError("web_search 没有返回可引用的搜索结果。")

references = [
    Reference(title=item["title"], url=item["url"])
    for item in search_items
]
search_context = [
    {
        "title": item["title"],
        "url": item["url"],
        "content": item.get("content", ""),
    }
    for item in search_items
]

response = agent.invoke({
    "messages": [HumanMessage(content=(
        f"用户问题：{question}\n\n"
        "下面是 web_search 已经实际返回的搜索结果。请根据这些结果直接作答；"
        "若问题无法从结果中确认具体作品或人气排名，请明确说明，不要编造。\n"
        f"搜索结果：{json.dumps(search_context, ensure_ascii=False)}"
    ))]
})

answer = response["structured_response"]
if not answer.answer.strip():
    raise RuntimeError("模型未能基于搜索结果生成有效回答。")

# 引用只取自本次搜索结果，不依赖模型自行编造引用。
answer = AnswerInfo(answer=answer.answer, reference=references)
print(f"web_search 已执行，搜索词：{search_result.get('query', question)}")
print(f"返回 {len(search_items)} 条可引用结果：")
for reference in answer.reference:
    print(f"- {reference.title}: {reference.url}")
print("\n结构化回答：")
print(answer)

web_search 已执行，搜索词：聊聊学生会也有洞里的角色，以及目前谁最受欢迎？
返回 5 条可引用结果：
- 学生会也有洞！ - 维基百科，自由的百科全书: https://zh.wikipedia.org/zh-hans/%E5%AD%B8%E7%94%9F%E6%9C%83%E4%B9%9F%E6%9C%89%E6%B4%9E%EF%BC%81
- 97K views · 1.9K reactions | 《學生會也有洞》這作品真的很有趣w
角色設計真的很棒 又色又可愛
不少雖然是配角但也很印象深刻
　
『生徒会にも穴はある！』
@seitokainoana | 阿布拉棒棒雞庇護所: https://www.facebook.com/100090260462772/videos/%E5%AD%B8%E7%94%9F%E6%9C%83%E4%B9%9F%E6%9C%89%E6%B4%9E%E9%80%99%E4%BD%9C%E5%93%81%E7%9C%9F%E7%9A%84%E5%BE%88%E6%9C%89%E8%B6%A3w-%E8%A7%92%E8%89%B2%E8%A8%AD%E8%A8%88%E7%9C%9F%E7%9A%84%E5%BE%88%E6%A3%92-%E5%8F%88%E8%89%B2%E5%8F%88%E5%8F%AF%E6%84%9B-%E4%B8%8D%E5%B0%91%E9%9B%96%E7%84%B6%E6%98%AF%E9%85%8D%E8%A7%92%E4%BD%86%E4%B9%9F%E5%BE%88%E5%8D%B0%E8%B1%A1%E6%B7%B1%E5%88%BB/889852759614450
- 脑洞学生会 - 萌娘百科 万物皆可萌的百科全书: https://zh.moegirl.org.cn/%E8%84%91%E6%B4%9E%E5%AD%A6%E7%94%9F%E4%BC%9A
- "Brain Hole Student Council Has Holes Too" all 12 episodes, updated weekly_Bilibili: https://www.bilibili.com/video/BV1SeHi63ENg
- YA.3：《學生會也有洞》《容易對付的惡魔大人》等作品動畫